In [37]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

from pathlib import Path
from dotenv import load_dotenv
from collections import defaultdict
from tqdm.notebook import tqdm


import logging
import pdf_logging as pf
import sections as sf
import embedding as eb
import os, json

from google import genai

try:
    load_dotenv()
    API_KEY = os.getenv("GOOGLE_API_KEY")
except Exception as e:
    print(f"Error loading environment variable(s): {e}")

PROJECT_ROOT = Path.cwd().parent
PDF_DIR = PROJECT_ROOT / "Data" / "financebench" / "pdfs"
JSON_DIR = PROJECT_ROOT / "Data" / "cache" / "pages"
CHROMA_DIR = PROJECT_ROOT / "Data" / "chroma"

logger = logging.getLogger("pypdf")
logger.setLevel(logging.WARNING)
logger.propagate = False # Prevent propagation to the root logger, to not get wierd log and console dupes

log_file = PROJECT_ROOT / "Data" / "logs" /"pypdf_warnings.json"
handler = logging.FileHandler(log_file)
handler.setFormatter(pf.JsonFormatter())
logger.addHandler(handler)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## I. Ingestion and chunking

In [38]:
from ingestion import load_pdfs
from datasets import load_dataset

load_dotenv()
meta = {
    r["doc_name"]: {"company_name": r["company"], "doc_type": r["doc_type"], "year": r["doc_period"]}
    for r in load_dataset("PatronusAI/financebench", split="train")
}
documents = load_pdfs(
    sorted(PDF_DIR.glob("*.pdf")),
    meta=meta,
    log_file=log_file,
    cache_dir=PROJECT_ROOT / "Data" / "cache" / "pages",
)
print(f"Loaded {len(documents)} pages.")
print(f"Preview:\n{documents[0].page_content[:200]}")

loading pdfs:   0%|          | 0/2 [00:00<?, ?it/s]

skipped INTEL_2023_8K_dated-2023-08-16.pdf: PdfStreamError: Stream has ended unexpectedly
skipped INTEL_2023_8K_dated-2023-02-10.pdf: PdfStreamError: Stream has ended unexpectedly


Loaded 53901 pages.
Preview:
Table of Contents
 
 
low
 
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
 
FORM 10-K
 
☒
   ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE
SECURITIES EXCHANGE ACT OF 1


## II. Sectionning and embedding

In [39]:
by_doc = defaultdict(list)
for d in documents:
    by_doc[d.metadata["doc_name"]].append(d)

all_sections, report = [], []
for name, pgs in tqdm(sorted(by_doc.items()), desc="sectioning"):
    pgs = sorted(pgs, key=lambda d: d.metadata["page"])
    form = sf.form_of(name, pgs[0].metadata.get("doc_type"))
    try:
        secs = sf.split_by_section(pgs)
    except Exception as e:
        logging.getLogger(__name__).exception("sectioning failed for %s", name)
        report.append((name, form, len(pgs), None, repr(e)))
        continue
    n_titled = sum(1 for s in secs if s.metadata.get("section_title"))
    report.append((name, form, len(pgs), len(secs), n_titled))
    all_sections.extend(secs)

print(f"{len(by_doc)} docs -> {len(all_sections)} sections")
failed = [r for r in report if r[3] is None]
flat = [r for r in report if r[3] is not None and r[4] == 0]
print(f"failed: {len(failed)}  | no titled section (heading detection found nothing): {len(flat)}")
for r in failed + flat:
    print("  ", r)

all_pieces = eb.prepare(all_sections)
print(f"{len(all_sections)} sections -> {len(all_pieces)} pieces to embed")

sectioning:   0%|          | 0/366 [00:00<?, ?it/s]

3M_2018_10K page 65 piece 0 is 6304 chars, may exceed the model input limit
3M_2018_10K page 66 piece 0 is 6220 chars, may exceed the model input limit
3M_2018_10K page 67 piece 0 is 6296 chars, may exceed the model input limit
3M_2018_10K page 68 piece 0 is 6193 chars, may exceed the model input limit
3M_2019_10K page 11 piece 0 is 6237 chars, may exceed the model input limit
3M_2019_10K page 61 piece 0 is 6036 chars, may exceed the model input limit
3M_2019_10K page 67 piece 0 is 6075 chars, may exceed the model input limit
3M_2019_10K page 110 piece 0 is 6173 chars, may exceed the model input limit
3M_2020_10K page 11 piece 0 is 6130 chars, may exceed the model input limit
3M_2020_10K page 62 piece 0 is 6083 chars, may exceed the model input limit
3M_2020_10K page 112 piece 0 is 6172 chars, may exceed the model input limit
3M_2020_10K page 115 piece 0 is 6038 chars, may exceed the model input limit
3M_2020_10K page 121 piece 0 is 6366 chars, may exceed the model input limit
3M_2021_

366 docs -> 58739 sections
failed: 0  | no titled section (heading detection found nothing): 56
   ('AMCOR_2023Q4_EARNINGS', None, 14, 14, 0)
   ('AMD_2022_annualreport', None, 122, 122, 0)
   ('BESTBUY_2023Q4_EARNINGS', None, 13, 13, 0)
   ('BESTBUY_2024Q2_EARNINGS', None, 12, 12, 0)
   ('BLOCK_2022_8K_dated-2022-01-28', '8-K', 133, 137, 0)
   ('COSTCO_2023Q1_EARNINGS', None, 4, 4, 0)
   ('COSTCO_2023Q3_EARNINGS', None, 4, 4, 0)
   ('EBAY_2023Q1_EARNINGS', None, 15, 15, 0)
   ('EBAY_2023Q2_EARNINGS', None, 15, 15, 0)
   ('FEDEX_2023_annualreport', None, 139, 139, 0)
   ('FOOTLOCKER_2023_annualreport', None, 100, 96, 0)
   ('GENERALMILLS_2023_annualreport', None, 99, 98, 0)
   ('INTEL_2017_10K', '10-K', 185, 185, 0)
   ('INTEL_2018_10K', '10-K', 138, 138, 0)
   ('INTEL_2019_10K', '10-K', 129, 129, 0)
   ('INTEL_2020_10K', '10-K', 122, 120, 0)
   ('INTEL_2021_10K', '10-K', 123, 121, 0)
   ('INTEL_2022Q4_EARNINGS', None, 17, 17, 0)
   ('INTEL_2022_10K', '10-K', 134, 132, 0)
   ('INTEL_20

3M_2022_10K page 94 piece 0 is 6133 chars, may exceed the model input limit
3M_2022_10K page 95 piece 0 is 6438 chars, may exceed the model input limit
3M_2022_10K page 99 piece 0 is 6691 chars, may exceed the model input limit
3M_2022_10K page 100 piece 0 is 6149 chars, may exceed the model input limit
3M_2022_10K page 105 piece 0 is 6242 chars, may exceed the model input limit
3M_2022_10K page 107 piece 0 is 6255 chars, may exceed the model input limit
3M_2022_10K page 108 piece 0 is 6345 chars, may exceed the model input limit
3M_2022_10K page 109 piece 0 is 6295 chars, may exceed the model input limit
3M_2022_10K page 112 piece 0 is 6908 chars, may exceed the model input limit
3M_2022_10K page 113 piece 0 is 6061 chars, may exceed the model input limit
3M_2023Q2_10Q page 29 piece 0 is 6032 chars, may exceed the model input limit
3M_2023Q2_10Q page 33 piece 0 is 6508 chars, may exceed the model input limit
3M_2023Q2_10Q page 37 piece 0 is 6305 chars, may exceed the model input limit

58739 sections -> 56247 pieces to embed


In [40]:
chars = sum(len(p.embed_text) for p in all_pieces)
pieces = len(all_pieces)
tokens = chars / 3.5
print(len(all_pieces), chars, tokens)

56247 181101396 51743256.0


In [45]:
embedding_client = genai.Client(api_key=API_KEY)
collection = eb.get_collection(CHROMA_DIR)

n = eb.index(collection, embedding_client, all_pieces, batch_size=20)
print(f"Embedded {n} pieces into collection, now holds {collection.count()} items.")

indexing:   0%|          | 0/2813 [00:00<?, ?it/s]

Embedded 35887 pieces into collection, now holds 56247 items.


In [47]:
request = "What were 3M's net sales in 2015?"
hits = eb.search(
    collection, embedding_client,
    request,
    k=5,
    where={"company_name": "3M"},
)
print(f"Search for {request!r} returned {len(hits)} hits:")
for h in hits:
    md = h["metadata"]
    print(f"{h['distance']:.3f}  p{md['page']} (label {md.get('page_label')})  {md.get('section_title')}  {h['text'][:80]!r}")

Search for "What were 3M's net sales in 2015?" returned 5 hits:
0.286  p13 (label 14)  Selected Financial Data  'Item 6. \nSelected Financial Dat\n\na.\n\n(Dollars in millions, except per share amou'
0.288  p72 (label 73)  Financial Statements and Supplementary Data  'Year ended December 31, 2018\n\nNet Sales (Millions)\n\nUnited\nStates\n\nAsia Pacific\n'
0.288  p59 (label 60)  Financial Statements and Supplementary Data  'T\nable of \nContents\nDisaggregated revenue information:\nThe Company views the fol'
0.305  p71 (label 72)  Financial Statements and Supplementary Data  'relating to the accelerated recognition for software installation service and tr'
0.311  p24 (label 25)  Management's Discussion and Analysis  'T\nable of \nContents\nSales and operating income (loss) by business segment:\nThe f'


In [50]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

template = """
Answer the question based on the context below:
{context}
Question: {question}.

In case the answer is not contained within the context, you have to mention it.
"""

def retrieve(question: str) -> str:
    hits = eb.search(collection, embedding_client, question, k = 5)
    return "\n\n".join([h['text'] for h in hits])

prompt = ChatPromptTemplate.from_messages([
    ("user", template)
])

llm = ChatGoogleGenerativeAI(model = "gemini-3.5-flash", temperature = 0)
rag_llm_chain = (
    {"context": retrieve, "question" : RunnablePassthrough()}
    | prompt 
    | llm
    | StrOutputParser()
)

rag_llm_chain.invoke(request)

"Based on the provided context, 3M's net sales in 2015 were **$30,274 million** (or approximately **$30.274 billion**)."